# API Queries - Strategy 2 Validation

This notebook validates API responses against base data for the Strategy 2 architecture.

## Current API Structure

**Base Scenarios** (3):
- `beslutad-policy` (default)
- `internationell-tillvaxt`
- `lokal-miljohansyn`

**Independent Parameters** (10):
- Growth: `housing_growth`, `transport_growth`, `industry_growth`, `services_growth`, `datacenters_growth`
- Flexibility: `housing_flex`, `transport_flex`, `industry_flex`, `services_flex`, `datacenters_flex`

**Segments** (5): housing, transport, industry, services, datacenters

**Data Files**:
- Base: `/api/data/base/{scenario-slug}/{segment}/data.parquet`
- Parameters: `/api/data/parameters/{param}/{index}/{segment}/data.parquet`
- Aggregated: `/api/data/aggregated/*.parquet`

In [ ]:
# Setup
import duckdb
import pandas as pd
import requests
import json
from pathlib import Path

# Paths
project_root = Path().resolve().parent.parent
api_data = project_root / 'api' / 'data'
base_dir = api_data / 'base'
params_dir = api_data / 'parameters'
agg_dir = api_data / 'aggregated'

# API base URL
API_URL = 'http://localhost:4010'

# DuckDB connection
db = duckdb.connect(':memory:')

print(f"Project root: {project_root}")
print(f"API data: {api_data}")
print(f"Base scenarios: {[d.name for d in base_dir.iterdir() if d.is_dir()]}")
print(f"Parameters: {[d.name for d in params_dir.iterdir() if d.is_dir()]}")

## 1. Static Endpoints

Test the static JSON endpoints that serve configuration data.

In [ ]:
# Static Endpoints
def fetch_endpoint(endpoint):
    """Fetch JSON from API endpoint"""
    try:
        r = requests.get(f"{API_URL}{endpoint}", timeout=5)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f"Error fetching {endpoint}: {e}")
        return None

# GET /scenarios
print("=== GET /scenarios ===")
scenarios = fetch_endpoint('/scenarios')
if scenarios:
    for s in scenarios:
        print(f"  - {s['id']}: {s['name']} {'(default)' if s.get('default') else ''}")

# GET /parameters (Strategy 2 config)
print("\n=== GET /parameters ===")
params = fetch_endpoint('/parameters')
if params:
    print(f"  Years: {params.get('years', [])[:3]}...{params.get('years', [])[-3:]}")
    print(f"  Geographies: {len(params.get('geographies', []))} regions")
    print(f"  Segments: {params.get('segments', [])}")
    if 'strategy2' in params:
        s2 = params['strategy2']
        print(f"  Strategy 2:")
        print(f"    Base scenarios: {len(s2.get('baseScenarios', []))}")
        print(f"    Parameters: {list(s2.get('parameters', {}).keys())}")

# GET /globals
print("\n=== GET /globals ===")
globals_data = fetch_endpoint('/globals')
if globals_data:
    print(f"  Bounds: {globals_data.get('bounds', {})}")

## 2. Base Data Validation

Query the raw parquet files to understand the base data.

In [ ]:
# Base Data Overview
print("=== Base Data Overview ===")

for scenario_dir in sorted(base_dir.iterdir()):
    if not scenario_dir.is_dir():
        continue
    
    print(f"\nScenario: {scenario_dir.name}")
    
    for segment_dir in sorted(scenario_dir.iterdir()):
        if not segment_dir.is_dir():
            continue
        
        parquet_file = segment_dir / 'data.parquet'
        if parquet_file.exists():
            result = db.execute(f"""
                SELECT 
                    COUNT(*) as rows,
                    COUNT(DISTINCT geography) as geos,
                    MIN(timestamp) as min_ts,
                    MAX(timestamp) as max_ts,
                    SUM(value) as total,
                    AVG(value) as avg_val
                FROM read_parquet('{parquet_file}')
            """).fetchone()
            
            print(f"  {segment_dir.name}: {result[0]:,} rows, {result[1]} geos, "
                  f"total={result[4]:,.0f} GWh, avg={result[5]:.2f}")

In [ ]:
# Yearly National Totals (All Segments Combined)
print("=== Yearly National Totals (GWh) ===")

base_scenario = 'beslutad-policy'
segments = ['housing', 'transport', 'industry', 'services', 'datacenters']

# Build UNION query for all segments
union_parts = []
for seg in segments:
    parquet_file = base_dir / base_scenario / seg / 'data.parquet'
    if parquet_file.exists():
        union_parts.append(f"""
            SELECT timestamp, geography, '{seg}' as segment, value
            FROM read_parquet('{parquet_file}')
        """)

if union_parts:
    union_query = " UNION ALL ".join(union_parts)
    
    yearly_query = f"""
        SELECT 
            EXTRACT(year FROM timestamp) as year,
            SUM(value) as total_gwh
        FROM ({union_query})
        GROUP BY EXTRACT(year FROM timestamp)
        ORDER BY year
    """
    
    yearly_df = db.execute(yearly_query).df()
    yearly_df['total_twh'] = yearly_df['total_gwh'] / 1000
    display(yearly_df)

## 3. Demand Endpoint Validation

Compare API `/demand` responses with raw data queries.

In [ ]:
# GET /demand - Yearly National Totals
print("=== GET /demand (Yearly, Total, Total) ===")

params = {
    'period[start]': '2025-01-01',
    'period[end]': '2051-01-01',
    'period[resolution]': '1Y',
    'period[aggregation]': 'sum',
    'geography': 'total',
    'segment': 'total',
    'baseScenario': 'beslutad-policy'
}

try:
    r = requests.get(f"{API_URL}/demand", params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    
    df = pd.DataFrame(data)
    df['period'] = pd.to_datetime(df['period'])
    df['year'] = df['period'].dt.year
    df['value_twh'] = df['value'] / 1000
    
    print(f"API returned {len(df)} rows")
    print(f"\nYear range: {df['year'].min()} - {df['year'].max()}")
    print(f"Value range: {df['value'].min():,.0f} - {df['value'].max():,.0f} GWh")
    print(f"           = {df['value_twh'].min():,.0f} - {df['value_twh'].max():,.0f} TWh")
    
    display(df[['year', 'value', 'value_twh', 'geography', 'segment', 'scenario_id']].head(10))
    
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# GET /demand - By Geography (for Map)
print("=== GET /demand (2030, All Geographies) ===")

params = {
    'period[start]': '2030-01-01',
    'period[end]': '2031-01-01',
    'period[resolution]': '1Y',
    'period[aggregation]': 'sum',
    'geography': 'all',
    'segment': 'total',
    'baseScenario': 'beslutad-policy'
}

try:
    r = requests.get(f"{API_URL}/demand", params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    
    df = pd.DataFrame(data)
    df['value_twh'] = df['value'] / 1000
    
    print(f"API returned {len(df)} geographies")
    display(df[['geography', 'value', 'value_twh']].sort_values('geography'))
    
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# GET /demand - By Segment
print("=== GET /demand (2030, All Segments) ===")

params = {
    'period[start]': '2030-01-01',
    'period[end]': '2031-01-01',
    'period[resolution]': '1Y',
    'period[aggregation]': 'sum',
    'geography': 'total',
    'segment': 'all',
    'baseScenario': 'beslutad-policy'
}

try:
    r = requests.get(f"{API_URL}/demand", params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    
    df = pd.DataFrame(data)
    df['value_twh'] = df['value'] / 1000
    
    print(f"API returned {len(df)} segments")
    display(df[['segment', 'value', 'value_twh']].sort_values('value', ascending=False))
    
except Exception as e:
    print(f"Error: {e}")

## 4. Strategy 2 Parameters

Test the independent parameter system.

In [ ]:
# Parameter Files Overview
print("=== Parameter Files ===")

for param_dir in sorted(params_dir.iterdir()):
    if not param_dir.is_dir():
        continue
    
    print(f"\n{param_dir.name}:")
    
    for index_dir in sorted(param_dir.iterdir()):
        if not index_dir.is_dir():
            continue
        
        segments_found = []
        for seg_dir in index_dir.iterdir():
            if seg_dir.is_dir() and (seg_dir / 'data.parquet').exists():
                segments_found.append(seg_dir.name)
        
        if segments_found:
            print(f"  Index {index_dir.name}: {segments_found}")

In [ ]:
# Sample Parameter Multipliers
print("=== Sample Parameter Values (housing_growth, index 2) ===")

param_file = params_dir / 'housing_growth' / '2' / 'housing' / 'data.parquet'
if param_file.exists():
    result = db.execute(f"""
        SELECT 
            EXTRACT(year FROM timestamp) as year,
            AVG(value) as avg_multiplier,
            MIN(value) as min_mult,
            MAX(value) as max_mult
        FROM read_parquet('{param_file}')
        GROUP BY EXTRACT(year FROM timestamp)
        ORDER BY year
    """).df()
    
    print("Yearly average multipliers:")
    display(result)
else:
    print(f"File not found: {param_file}")

In [ ]:
# GET /demand with Parameter Adjustments
print("=== GET /demand with housing_growth=2 ===")

# Baseline query
params_baseline = {
    'period[start]': '2025-01-01',
    'period[end]': '2051-01-01',
    'period[resolution]': '1Y',
    'period[aggregation]': 'sum',
    'geography': 'total',
    'segment': 'total',
    'baseScenario': 'beslutad-policy'
}

# With parameter
params_adjusted = {
    **params_baseline,
    'housing_growth': '2'
}

try:
    r1 = requests.get(f"{API_URL}/demand", params=params_baseline, timeout=30)
    r2 = requests.get(f"{API_URL}/demand", params=params_adjusted, timeout=30)
    
    df1 = pd.DataFrame(r1.json())
    df2 = pd.DataFrame(r2.json())
    
    df1['year'] = pd.to_datetime(df1['period']).dt.year
    df2['year'] = pd.to_datetime(df2['period']).dt.year
    
    # Compare
    comparison = df1[['year', 'value']].merge(
        df2[['year', 'value']], 
        on='year', 
        suffixes=('_baseline', '_adjusted')
    )
    comparison['diff_pct'] = (comparison['value_adjusted'] - comparison['value_baseline']) / comparison['value_baseline'] * 100
    
    print("Baseline vs housing_growth=2:")
    display(comparison.head(10))
    
except Exception as e:
    print(f"Error: {e}")

## 5. Aggregated Tables

Validate pre-computed aggregated tables.

In [ ]:
# Aggregated Tables Overview
print("=== Aggregated Tables ===")

agg_files = {
    'national_yearly': agg_dir / 'national_yearly.parquet',
    'geography_yearly': agg_dir / 'geography_yearly.parquet',
    'segment_yearly': agg_dir / 'segment_yearly.parquet'
}

for name, filepath in agg_files.items():
    if filepath.exists():
        result = db.execute(f"""
            SELECT 
                COUNT(*) as rows,
                COUNT(DISTINCT scenario_id) as scenarios,
                MIN(year) as min_year,
                MAX(year) as max_year
            FROM read_parquet('{filepath}')
        """).fetchone()
        print(f"{name}: {result[0]} rows, {result[1]} scenarios, years {result[2]}-{result[3]}")
    else:
        print(f"{name}: NOT FOUND")

In [ ]:
# National Yearly from Aggregated Table
print("=== National Yearly (from aggregated table) ===")

agg_file = agg_dir / 'national_yearly.parquet'
if agg_file.exists():
    df = db.execute(f"""
        SELECT 
            scenario_id,
            year,
            total_value as value_gwh,
            total_value / 1000 as value_twh
        FROM read_parquet('{agg_file}')
        WHERE scenario_id = 'Beslutad Policy'
        ORDER BY year
    """).df()
    
    display(df)

## 6. Data Validation

Compare API responses with direct parquet queries.

In [ ]:
# Validate: API vs Raw Data
print("=== Validation: API vs Raw Parquet ===")

# Get API data
params = {
    'period[start]': '2030-01-01',
    'period[end]': '2031-01-01',
    'period[resolution]': '1Y',
    'period[aggregation]': 'sum',
    'geography': 'total',
    'segment': 'total',
    'baseScenario': 'beslutad-policy'
}

try:
    r = requests.get(f"{API_URL}/demand", params=params, timeout=30)
    api_data = r.json()
    api_value = api_data[0]['value'] if api_data else 0
    print(f"API value (2030): {api_value:,.2f} GWh")
except Exception as e:
    print(f"API Error: {e}")
    api_value = None

# Get raw data
base_scenario = 'beslutad-policy'
segments = ['housing', 'transport', 'industry', 'services', 'datacenters']

union_parts = []
for seg in segments:
    parquet_file = base_dir / base_scenario / seg / 'data.parquet'
    if parquet_file.exists():
        union_parts.append(f"""
            SELECT value FROM read_parquet('{parquet_file}')
            WHERE timestamp >= '2030-01-01' AND timestamp < '2031-01-01'
        """)

if union_parts:
    raw_query = f"SELECT SUM(value) as total FROM ({' UNION ALL '.join(union_parts)})"
    raw_value = db.execute(raw_query).fetchone()[0]
    print(f"Raw value (2030): {raw_value:,.2f} GWh")
    
    if api_value:
        diff = abs(api_value - raw_value)
        diff_pct = diff / raw_value * 100
        print(f"\nDifference: {diff:,.2f} GWh ({diff_pct:.4f}%)")
        if diff_pct < 0.01:
            print("✅ VALIDATION PASSED")
        else:
            print("❌ VALIDATION FAILED - values differ significantly")

In [ ]:
# Cleanup
db.close()
print("Database connection closed.")